# Study 912 — Gold + Trend 🥇

**Does a 200-day trend filter turn gold into a better drawdown-managed diversifier?**

Gold is a volatile diversifier (~18% vol) with famously long *dead decades* — 1980-2001,
2012-2018. The folk fix, borrowed from Faber's tactical playbook: hold gold only when its
price is **above** its 200-day moving average, else sit in T-bills. The promise: duck the
dead decades, earn a **better excess-of-cash Sharpe** and **much shallower drawdowns** than
buy-and-hold gold.

We test it on **GLD vs BIL** (cash) daily total-return closes, 2007-05-30 → 2026-06-30
(4,802 days), every leg **excess-of-cash**, 5 bps one-way cost.

*Numbers below are the frozen headline (`docs/results.md`, Fingerprint `3ea3fb6649e8`); the one
live cell runs the fast synthetic control. As-of 2026-06-30.*


## 1. The idea in one picture

Gold spends years going nowhere, then years soaring. If a slow 200-day filter could keep you *in* for the soars and *out* for the dead decades, gold would become a smooth diversifier. Let's see whether it actually does.

In [1]:
R = dict(bh_sharpe=0.52, tr_sharpe=0.332, adv=-0.188, t_diff=-1.83,
         bh_dd=-45.56, tr_dd=-35.3, bh_cagr=8.08, tr_cagr=3.78)
print('Buy-and-hold gold : excess Sharpe %+.3f, CAGR %+.2f%%, worst DD %.1f%%'
      % (R['bh_sharpe'], R['bh_cagr'], R['bh_dd']))
print('200-day trend     : excess Sharpe %+.3f, CAGR %+.2f%%, worst DD %.1f%%'
      % (R['tr_sharpe'], R['tr_cagr'], R['tr_dd']))
print('advantage (trend - hold): Sharpe %+.3f  (HAC t = %+.2f)'
      % (R['adv'], R['t_diff']))

Buy-and-hold gold : excess Sharpe +0.520, CAGR +8.08%, worst DD -45.6%
200-day trend     : excess Sharpe +0.332, CAGR +3.78%, worst DD -35.3%
advantage (trend - hold): Sharpe -0.188  (HAC t = -1.83)


## 2. The surprise — the overlay makes gold *worse* risk-adjusted

The trend overlay cuts the worst drawdown from **-45.6%** to **-35.3%** (a 10 pp cushion) — but it does it by forfeiting so much return (CAGR **8.1% → 3.8%**) that its **excess-of-cash Sharpe falls** (0.520 → 0.332). The advantage is **-0.188** with the *wrong sign* (HAC *t* = -1.83). Why? Gold's biggest rallies are sharp V-shaped recoveries off the lows — and a 200-day filter always re-enters them late.

## 3. But it's not just random luck — it *is* genuine timing

A random control that goes to cash on *random* days at the same 63% in-market rate does far worse (Sharpe 0.142, DD -55.3%). So the trend rule really is picking *better* days than chance (+0.190 vs random) — it's just that even good timing on gold under-performs simply holding gold. Genuine skill, wrong asset.

## 4. And the drawdown cushion doesn't last

Split the sample in 2016: the whole drawdown benefit comes from the *early* era (ducking the 2013 gold crash, DD -46% → -35%). In the *recent* era the overlay's drawdown is actually **deeper** than just holding gold (-26% → -30%). The advantage is negative in **both** halves (-0.28 / -0.20).

## 5. Live check — the machinery is unbiased (offline synthetic)

On a *planted* dead-decade world the overlay works as advertised (cuts drawdown, lifts Sharpe); on a no-regime null it does nothing. So the real-tape miss is a fact about *gold*, not a broken backtest.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from gold_trend import data, strategy as st
planted = st.synthetic_detect(data.synthetic_daily(signal_strength=1.0, seed=912)[0])
null    = st.synthetic_detect(data.synthetic_daily(signal_strength=0.0, seed=912)[0])
print('planted dead decade: DD improvement %+.1f pp, excess-Sharpe adv %+.2f (should help)'
      % ((planted['dd_improvement'])*100, planted['excess_sharpe_adv']))
print('flat-vol null      : excess-Sharpe adv %+.2f (should be ~0)'
      % null['excess_sharpe_adv'])

planted dead decade: DD improvement +23.4 pp, excess-Sharpe adv +0.14 (should help)
flat-vol null      : excess-Sharpe adv -0.19 (should be ~0)


## Verdict

- **Signal — None.** The claimed *better-Sharpe* drawdown-managed gold does not replicate: excess-Sharpe advantage **-0.188** (wrong sign, HAC *t* = -1.83), CI includes zero, negative in both eras, and the drawdown cushion reverses after 2016.
- **Tradability — Mirage.** Negative gross, more negative with cost. The overlay buys a smaller (and recently absent) worst-loss with ~4.3 pp/yr of CAGR — a bad trade. Hold gold and size it; don't trend-time it.